In [6]:
import torch
from torch.utils.data import Dataset, DataLoader, TensorDataset

import numpy as np
import matplotlib.pyplot as plt
from PIL import Image, ImageDraw

import os
import numpy as np
import random
import matplotlib.pyplot as plt

import torch
import torch.nn as nn
import torchvision
import torchvision.transforms as transforms
from torchvision.datasets.utils import download_url

In [ ]:
transform = transforms.Compose([transforms.ToTensor(), transforms.Normalize((0.5,), (0.5,))])
download_url('https://biometrics.nist.gov/cs_links/EMNIST/gzip.zip', root='/Users/canelatustin/Desktop/SSRP', filename="emnist.zip", md5='58c8d27c78d21e728a6bc7b3cc06412e')

In [ ]:
trainset = torchvision.datasets.MNIST(root='/Users/canelatustin/Desktop/MNIST', train=True, download=True, transform=transform
testset = torchvision.datasets.MNIST(root='/Users/canelatustin/Desktop/MNIST', train=False, download=True, transform=transform)

In [ ]:
batch_size = 50
trainloader = torch.utils.data.DataLoader(trainset, batch_size=batch_size, shuffle=False)
testloader = torch.utils.data.DataLoader(testset, batch_size=batch_size, shuffle=False)

In [ ]:
trainset.data.shape

In [ ]:
# we enter input of the form [batch_size, 1, 28, 28] and we output a tensor of size [batch_size, 62]
class CNN(nn.Module):
    def __init__(self):
        super(CNN, self).__init__() #just makes it so you can initialize them both implicitly
        self.model = nn.Sequential( 
            nn.Conv2d(1,400, 5, 2, 0), #torch.nn.Conv2d(in_channels, out_channels, kernel_size, stride=1, padding=0, dilation=1, groups=1, bias=True, padding_mode='zeros', device=None, dtype=None)
            nn.LeakyReLU(0.2, inplace=True), #first term is the negative slope, inplace means it deletes the original value of the input and reassigns it to whatever the output (saves a little memory, useful for sequential)
            nn.Conv2d(400,100, 5, 2, 0),
            nn.ReLU(),
            nn.Conv2d(100,80, 3, 1, 0), #to calculate the dimensions of output: [(W−K+2P)/S]+1 where [] is floor and W is choice input dim, S = stride length, K = kernel size,P=padding
            nn.LeakyReLU(0.2, inplace=True), 
            nn.Flatten(1), #flattens everything - (start_dim=1, end_dim = -1), so here 1 means start flattening at the first dimension
            nn.Linear(320,100), #torch.nn.Linear(in_features, out_features, bias=True, device=None, dtype=None) in and out_features are number of columns of input and output
            nn.ReLU(),
            nn.Linear(100,62),
            nn.Sigmoid()
        )
    def forward(self, input):
        return self.model(input)

In [ ]:
net = CNN()

optimizer = torch.optim.Adam(net.parameters(), lr=0.0002, betas=(0.5, 0.999)) 
#lr = learning rate, betas = coefs used when calculating averages of gradient etc, net.parameters() are variables to optimize (weights and biases); 
criterion = nn.MSELoss() 
# finds mean squared error
if not os.path.exists('output'):
    os.makedirs('output')

In [ ]:
losses = []
epochs = 7

for epoch in range(epochs):
    j = 0
    for batch, label in trainloader:
        j += 1
        net.zero_grad()

        # create the labels tensor; we compare the output to label probs
        label_probs = torch.zeros((batch_size, 62))
        for i in range(batch_size):
            label_probs[i,label[i]] = 1
        
        output = net(batch)
        loss = criterion(output, label_probs)
        #net(batch) is calling the model on one batch of data and outputting it; then you calculate loss between this and the labels of 1
        
        loss.backward()
        optimizer.step()        # net.zero_grad, loss.backward(), and optimizer.step() are the necessary steps for backpropagation 
                                # (reset gradients, do backpropagation, set new parameters)
        
        print('Epoch %d/%d; batch %d/%d; loss %.4f' % (epoch, epochs, j, len(trainloader), loss))
    
    losses.append(loss.item()) #append the loss for each epoch
print(label_probs)

In [ ]:
# save the weights of the trained network
torch.save(net.state_dict(), 'output/net.pth')

# save the loss values during training
np.save('output/losses', np.array(losses))

In [ ]:
#NEED TO ADD PREDICTOR AND COLLAPSE PREVENTOR -- AND TARGET ENCODER